# NHSSynth Comprehensive Example - Hypertension Dataset

This notebook demonstrates the **complete NHSSynth workflow** using all available modules:

**What you'll learn:**
- Loading and transforming data with optimized GMM settings
- Training a VAE model with KL annealing and free bits
- Monitoring training to prevent posterior collapse
- Generating synthetic data with adaptive temperature scaling
- Comprehensive evaluation using SDMetrics
- Advanced visualizations (t-SNE, distribution overlays)

**Modules demonstrated:**
- ✅ DataLoader: MetaData, MetaTransformer
- ✅ Model: VAE with optimized training
- ✅ Plotting: t-SNE, distribution overlays
- ✅ Evaluation: SDMetrics quality and diagnostic reports

## Setup

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Add the src directory to path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../src")))

from sdmetrics.reports.single_table import DiagnosticReport, QualityReport

from nhssynth.modules.dataloader.metadata import MetaData
from nhssynth.modules.dataloader.metatransformer import MetaTransformer
from nhssynth.modules.model.models import VAE
from nhssynth.modules.plotting.plots import tsne

# Optional: visualize constraint graph
try:
    import gravis as gv

    HAS_GRAVIS = True
except ImportError:
    HAS_GRAVIS = False
    print("Note: Install 'gravis' to visualize constraint graphs: pip install gravis")

print("✅ All modules imported successfully")

## 1. Load Data

In [ ]:
# Load original dataset
dataset = pd.read_csv("../data/hypertension_synthetic.csv")

# Convert datetime columns
dataset["Date_of_Birth"] = pd.to_datetime(dataset["Date_of_Birth"], errors="coerce")
dataset["Medication_Start_Date"] = pd.to_datetime(dataset["Medication_Start_Date"], errors="coerce")
dataset["Last_Followup_Date"] = pd.to_datetime(dataset["Last_Followup_Date"], errors="coerce")

print(f"Dataset shape: {dataset.shape}")
print(f"\nColumns: {list(dataset.columns)}")
print("\nSample statistics:")
print(dataset[["Date_of_Birth", "Obesity", "Baseline_BP_Systolic"]].describe())

## 2. Load Metadata and Create Transformer

In [ ]:
# Load metadata and create transformer
md = MetaData.from_path(dataset, "../data/hypertension_metadata.yaml")
mt = MetaTransformer(dataset, md)

print("\nConstraints loaded:")
print(mt._metadata.constraints.minimal_constraints)

# Optional: Visualize constraint graph
if HAS_GRAVIS:
    display(gv.d3(mt._metadata.constraints.minimal_graph))

## 3. Transform Data

In [ ]:
# Apply transformation
transformed_dataset = mt.apply()

print(f"\nTransformed shape: {transformed_dataset.shape}")
print(f"Original shape: {dataset.shape}")
print(f"\n📝 NOTE: Column count increased from {dataset.shape[1]} to {transformed_dataset.shape[1]}")
print("   This is TEMPORARY for VAE training due to one-hot encoding.")
print("   For example:")
print("   - Gender (binary: 0,1) → Gender_0, Gender_1 (2 columns)")
print("   - Ethnicity (5 categories: 0-4) → Ethnicity_0 through Ethnicity_4 (5 columns)")
print(f"\n   The VAE trains on this expanded {transformed_dataset.shape[1]}-column format.")
print(f"   During generation, model.generate() automatically reverts back to {dataset.shape[1]} columns.")

## 4. Train VAE Model

**Training configuration:**
- KL Annealing: beta 0.0 → 1.0 over 100 epochs
- Free Bits: 2.0 nats/dimension
- 200 epochs total

In [ ]:
# Create and train VAE
model = VAE(transformed_dataset, mt)

stats = model.train(
    notebook_run=True,
    num_epochs=200,
    patience=999,  # Train all epochs
)

print(f"\nTraining completed: {stats[0]} epochs")
print(f"Final KLD: {stats[1]['KLD'][-100:].mean():.2f}")

# Check for posterior collapse
final_kld = stats[1]["KLD"][-100:].mean()
if final_kld < 10:
    print("\n⚠️  WARNING: Posterior collapse detected")
elif final_kld > 500:
    print("\n⚠️  Note: High KLD, may be over-regularized")
else:
    print(f"\n✅ Healthy KLD ({final_kld:.1f})")

## 4.1. Visualize Training Curves

In [ ]:
# Plot training diagnostics
model.plot_training_curves(save_path="../experiments/vae_training_hypertension.png")
print("\n✅ Training curves saved to: experiments/vae_training_hypertension.png")

## 5. Generate Synthetic Data

In [ ]:
# Generate synthetic data
synthetic_dataset = model.generate()

print(f"\nSynthetic dataset shape: {synthetic_dataset.shape}")
print(f"✓ Correctly reverted from {transformed_dataset.shape[1]} to {synthetic_dataset.shape[1]} columns")

# Diagnostic checks
print(f"\n{'='*60}")
print("DIAGNOSTIC CHECKS")
print(f"{'='*60}")

# Check for NaN values
nan_cols = synthetic_dataset.columns[synthetic_dataset.isna().any()].tolist()
if nan_cols:
    print(f"\n⚠️  WARNING: NaN values detected in {len(nan_cols)} columns:")
    for col in nan_cols[:5]:  # Show first 5
        nan_count = synthetic_dataset[col].isna().sum()
        print(f"   - {col}: {nan_count} NaN ({nan_count/len(synthetic_dataset)*100:.1f}%)")
    if len(nan_cols) > 5:
        print(f"   ... and {len(nan_cols)-5} more columns")
else:
    print("\n✓ No NaN values detected")

# Check binary/categorical variables
print(f"\nBinary variable ranges (should be {0, 1}):")
binary_cols = [
    "Gender",
    "Hypertension_Diagnosis",
    "Diabetes",
    "Heart_Disease",
    "Obesity",
    "Chronic_Kidney_Disease",
    "Family_History_Hypertension",
]
for col in binary_cols:
    if col in synthetic_dataset.columns:
        unique_vals = synthetic_dataset[col].dropna().unique()
        min_val = synthetic_dataset[col].min()
        max_val = synthetic_dataset[col].max()
        if set(unique_vals).issubset({0, 1, 0.0, 1.0}):
            print(f"   ✓ {col}: {sorted(unique_vals)}")
        else:
            print(f"   ⚠️  {col}: range [{min_val:.3f}, {max_val:.3f}], unique count: {len(unique_vals)}")

print(f"\n{'='*60}\n")

print("First few rows:")
display(synthetic_dataset.head())

## 5.1. Diagnostic Checks

**What to look for:**
- ✅ **No NaN values**: All columns should have complete data
- ✅ **Binary variables are discrete**: Should only contain {0, 1}
- ✅ **Categorical variables in range**: E.g., Ethnicity should be {0,1,2,3,4}

**If you see issues:**
- **NaN values**: May indicate constraint repair failures or missing value handling issues
- **Continuous binary values**: Suggests the inverse transformation isn't properly discretizing
- **Out-of-range values**: Indicates constraint violations during generation

## 6. Distribution Overlay - All Variables

Compare all numeric variables at once for quick visual assessment.

In [ ]:
# Get all numeric columns
numeric_cols = dataset.select_dtypes(include=[np.number]).columns.tolist()

# Create grid plot
n_cols = 4
n_rows = (len(numeric_cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 4))
axes = axes.flatten() if n_rows > 1 else [axes]

for idx, col in enumerate(numeric_cols):
    if col in dataset.columns and col in synthetic_dataset.columns:
        axes[idx].hist(dataset[col].dropna(), bins=30, alpha=0.5, color="steelblue", label="Original", density=True)
        axes[idx].hist(
            synthetic_dataset[col].dropna(), bins=30, alpha=0.5, color="orange", label="Synthetic", density=True
        )
        axes[idx].set_title(col, fontsize=10, fontweight="bold")
        axes[idx].legend(fontsize=8)

# Hide unused subplots
for idx in range(len(numeric_cols), len(axes)):
    axes[idx].axis("off")

plt.suptitle("Distribution Overlay: All Numeric Variables", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig("../experiments/hypertension_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"✅ Plotted {len(numeric_cols)} numeric variables")

## 7. t-SNE Visualization

Visualize data similarity in reduced dimensional space.

In [ ]:
# t-SNE visualization using NHSSynth plotting module
print("Running t-SNE dimensionality reduction...")

try:
    # Handle NaN values before t-SNE
    dataset_clean = dataset.dropna()
    synthetic_clean = synthetic_dataset.dropna()

    if len(dataset_clean) < len(dataset) or len(synthetic_clean) < len(synthetic_dataset):
        print(
            f"⚠️  Dropped rows with NaN: Original {len(dataset)} → {len(dataset_clean)}, Synthetic {len(synthetic_dataset)} → {len(synthetic_clean)}"
        )

    if len(dataset_clean) > 0 and len(synthetic_clean) > 0:
        tsne(dataset_clean.copy(), synthetic_clean.copy())
        print("\n✅ t-SNE plot displayed")
        print("   Good overlap = similar distributions")
    else:
        print("⚠️  Insufficient data after removing NaN values")

except Exception as e:
    print(f"⚠️  t-SNE failed: {e}")
    import traceback

    traceback.print_exc()

# Generate quality report
print("Generating SDMetrics quality report...\n")

try:
    quality_report = QualityReport()
    # FIX: Don't pass metadata - let SDMetrics infer it from the actual data
    # The metadata object has transformed (one-hot encoded) columns, but the 
    # synthetic data has been reverted back to original structure
    quality_report.generate(dataset, synthetic_dataset, metadata=None)
    
    overall_score = quality_report.get_score()
    print(f"{'='*80}")
    print(f"OVERALL QUALITY SCORE: {overall_score:.1%}")
    print(f"{'='*80}\n")
    
    # Column Shapes
    properties = quality_report.get_properties()
    column_shapes = properties['Column Shapes']
    print(f"Column Shapes Score: {column_shapes['Score']:.1%}")
    
    details = pd.DataFrame(column_shapes['Details']['Column'])
    print(f"\nTop 5 Best Matching:")
    display(details.sort_values('Score', ascending=False).head().round(3))
    
    print(f"\nBottom 5 (Need Improvement):")
    display(details.sort_values('Score', ascending=False).tail().round(3))
    
    # Column Pair Trends
    column_pairs = properties['Column Pair Trends']
    print(f"\nColumn Pair Trends Score: {column_pairs['Score']:.1%}")
    
    # Visualizations
    fig = quality_report.get_visualization(property_name='Column Shapes')
    fig.show()
    
    print(f"\n✅ Quality assessment complete!")
    
except Exception as e:
    print(f"⚠️  Quality report failed: {e}")
    import traceback
    traceback.print_exc()
    overall_score = None
    column_shapes = None
    column_pairs = None

In [ ]:
# Generate quality report
print("Generating SDMetrics quality report...\n")

try:
    quality_report = QualityReport()
    # FIX: Use md.get_sdv_metadata() instead of md._metadata.get_sdv_metadata()
    quality_report.generate(dataset, synthetic_dataset, md.get_sdv_metadata())

    overall_score = quality_report.get_score()
    print(f"{'='*80}")
    print(f"OVERALL QUALITY SCORE: {overall_score:.1%}")
    print(f"{'='*80}\n")

    # Column Shapes
    properties = quality_report.get_properties()
    column_shapes = properties["Column Shapes"]
    print(f"Column Shapes Score: {column_shapes['Score']:.1%}")

    details = pd.DataFrame(column_shapes["Details"]["Column"])
    print("\nTop 5 Best Matching:")
    display(details.sort_values("Score", ascending=False).head().round(3))

    print("\nBottom 5 (Need Improvement):")
    display(details.sort_values("Score", ascending=False).tail().round(3))

    # Column Pair Trends
    column_pairs = properties["Column Pair Trends"]
    print(f"\nColumn Pair Trends Score: {column_pairs['Score']:.1%}")

    # Visualizations
    fig = quality_report.get_visualization(property_name="Column Shapes")
    fig.show()

    print("\n✅ Quality assessment complete!")

except Exception as e:
    print(f"⚠️  Quality report failed: {e}")
    import traceback

    traceback.print_exc()
    overall_score = None
    column_shapes = None
    column_pairs = None

# Generate diagnostic report
print("Generating diagnostic report...\n")

try:
    diagnostic_report = DiagnosticReport()
    # FIX: Don't pass metadata - let SDMetrics infer it from the actual data
    diagnostic_report.generate(dataset, synthetic_dataset, metadata=None)
    
    diagnostic_score = diagnostic_report.get_score()
    print(f"{'='*80}")
    print(f"DIAGNOSTIC SCORE: {diagnostic_score:.1%}")
    print(f"{'='*80}\n")
    
    properties = diagnostic_report.get_properties()
    data_validity = properties['Data Validity']
    data_structure = properties['Data Structure']
    
    print(f"Data Validity: {data_validity['Score']:.1%}")
    print(f"Data Structure: {data_structure['Score']:.1%}")
    
    # Show validity details
    validity_details = pd.DataFrame(data_validity['Details']['Column'])
    print(f"\nData Validity Details:")
    display(validity_details.head(10).round(3))
    
    # Visualizations
    fig = diagnostic_report.get_visualization(property_name='Data Validity')
    fig.show()
    
    print(f"\n✅ Diagnostic assessment complete!")
    
except Exception as e:
    print(f"⚠️  Diagnostic report failed: {e}")
    import traceback
    traceback.print_exc()
    diagnostic_score = None
    data_validity = None
    data_structure = None

In [ ]:
# Generate diagnostic report
print("Generating diagnostic report...\n")

try:
    diagnostic_report = DiagnosticReport()
    # FIX: Use md.get_sdv_metadata() instead of md._metadata.get_sdv_metadata()
    diagnostic_report.generate(dataset, synthetic_dataset, md.get_sdv_metadata())

    diagnostic_score = diagnostic_report.get_score()
    print(f"{'='*80}")
    print(f"DIAGNOSTIC SCORE: {diagnostic_score:.1%}")
    print(f"{'='*80}\n")

    properties = diagnostic_report.get_properties()
    data_validity = properties["Data Validity"]
    data_structure = properties["Data Structure"]

    print(f"Data Validity: {data_validity['Score']:.1%}")
    print(f"Data Structure: {data_structure['Score']:.1%}")

    # Show validity details
    validity_details = pd.DataFrame(data_validity["Details"]["Column"])
    print("\nData Validity Details:")
    display(validity_details.head(10).round(3))

    # Visualizations
    fig = diagnostic_report.get_visualization(property_name="Data Validity")
    fig.show()

    print("\n✅ Diagnostic assessment complete!")

except Exception as e:
    print(f"⚠️  Diagnostic report failed: {e}")
    import traceback

    traceback.print_exc()
    diagnostic_score = None
    data_validity = None
    data_structure = None

## 10. Summary and Export

In [ ]:
# Save synthetic data
output_path = "../data/hypertension_synthetic_generated.csv"
synthetic_dataset.to_csv(output_path, index=False)

# Print comprehensive summary
print("=" * 80)
print("COMPREHENSIVE EVALUATION SUMMARY")
print("=" * 80)
print("\nDataset: hypertension_synthetic")
print(f"Original size: {dataset.shape}")
print(f"Synthetic size: {synthetic_dataset.shape}")

print("\nTraining Configuration:")
print(f"  - Epochs: {stats[0]}")
print(f"  - Final KLD: {final_kld:.1f}")
print("  - Beta annealing: 0.0 → 1.0 over 100 epochs")
print("  - Free bits: 2.0 per dimension")

if overall_score is not None:
    print("\nQuality Metrics (SDMetrics):")
    print(f"  - Overall Quality: {overall_score:.1%}")
    print(f"  - Column Shapes: {column_shapes['Score']:.1%}")
    print(f"  - Column Pair Trends: {column_pairs['Score']:.1%}")

if diagnostic_score is not None:
    print("\nDiagnostic Metrics (SDMetrics):")
    print(f"  - Overall Diagnostic: {diagnostic_score:.1%}")
    print(f"  - Data Validity: {data_validity['Score']:.1%}")
    print(f"  - Data Structure: {data_structure['Score']:.1%}")

print("\n" + "=" * 80)
print("Generated Artifacts:")
print("=" * 80)
print(f"  ✓ Synthetic data: {output_path}")
print("  ✓ Training curves: experiments/vae_training_hypertension.png")
print("  ✓ Distribution overlays: experiments/hypertension_distributions.png")

print("\n" + "=" * 80)
print("EVALUATION COMPLETE!")
print("=" * 80)
print("\nNext Steps:")
print("  1. Review quality scores (target: >80% for production)")
print("  2. Inspect distribution overlays for visual similarity")
print("  3. Check t-SNE plot for overall distribution matching")
print("  4. Review column-specific scores for problematic variables")
print("  5. Iterate on metadata/constraints if needed\n")